In [1]:
import torch

print("PyTorch version:", torch.__version__)
print("Torch location:", torch.__file__)
print("MPS available:", torch.backends.mps.is_available())

PyTorch version: 2.6.0
Torch location: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/readabert_env/lib/python3.13/site-packages/torch/__init__.py
MPS available: True


In [2]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch

from scipy.stats import pearsonr, spearmanr
from torch.utils.data import Dataset, DataLoader
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification
)



# 1 Define project paths


BASE_DIR = Path.cwd()

DATA_DIR = Path(
    "/Users/nataliakolbina/Documents/workspace/haptic/"
    "CLEAR_original_grouped_split_2026-09-25"
)

TEST_PATH = (
    DATA_DIR / "CLEAR_original_test.csv"
)

TRAIN_PATH = (
    DATA_DIR / "CLEAR_original_train.csv"
)

MODEL_PATH = (
    BASE_DIR
    / "readabert_clear_finetuned_best_rerun"
)

OUTPUT_CSV = (
    BASE_DIR
    / "clear_test_readabert_final_rerun.csv"
)

ACTUAL_PREDICTED_PLOT = (
    BASE_DIR
    / "readabert_test_actual_vs_predicted_rerun.png"
)

DIFFICULTY_PLOT = (
    BASE_DIR
    / "readabert_difficulty_distribution_rerun.png"
)

print(
    "Test file exists:",
    TEST_PATH.exists()
)

print(
    "Training file exists:",
    TRAIN_PATH.exists()
)

print(
    "Model exists:",
    MODEL_PATH.exists()
)



# 2 Load the held-out CLEAR test set


test = pd.read_csv(
    TEST_PATH
)

test = test.dropna(
    subset=[
        "Excerpt",
        "BT.Easiness"
    ]
).copy()

test["Excerpt"] = (
    test["Excerpt"]
    .astype(str)
)

test["BT.Easiness"] = (
    test["BT.Easiness"]
    .astype(float)
)

print(
    "\nTest shape:",
    test.shape
)

print(
    test[
        [
            "Excerpt",
            "BT.Easiness"
        ]
    ].head()
)



# 3 Load the fully fine-tuned ReadaBERT checkpoint


tokenizer = (
    AutoTokenizer
    .from_pretrained(
        MODEL_PATH
    )
)

model = (
    AutoModelForSequenceClassification
    .from_pretrained(
        MODEL_PATH
    )
)

print(
    "\nModel loaded from:",
    MODEL_PATH
)



# 4 Select the available computing device


if torch.cuda.is_available():

    device = torch.device(
        "cuda"
    )

elif (
    hasattr(
        torch.backends,
        "mps"
    )
    and
    torch.backends.mps.is_available()
):

    device = torch.device(
        "mps"
    )

else:

    device = torch.device(
        "cpu"
    )

print(
    "Device:",
    device
)

model.to(
    device
)

model.eval()



# 5 Prepare the CLEAR test dataset


MAX_LENGTH = 288


class CLEARDataset(Dataset):

    def __init__(
        self,
        texts,
        labels,
        tokenizer,
        max_length
    ):
        self.texts = (
            texts.tolist()
        )

        self.labels = (
            labels.tolist()
        )

        self.tokenizer = (
            tokenizer
        )

        self.max_length = (
            max_length
        )

    def __len__(self):

        return len(
            self.texts
        )

    def __getitem__(
        self,
        idx
    ):
        encoding = self.tokenizer(
            self.texts[idx],
            truncation=True,
            padding="max_length",
            max_length=self.max_length,
            return_tensors="pt"
        )

        return {
            "input_ids":
                encoding[
                    "input_ids"
                ].squeeze(0),

            "attention_mask":
                encoding[
                    "attention_mask"
                ].squeeze(0),

            "labels":
                torch.tensor(
                    self.labels[idx],
                    dtype=torch.float
                )
        }


test_dataset = CLEARDataset(
    test["Excerpt"],
    test["BT.Easiness"],
    tokenizer,
    MAX_LENGTH
)

test_loader = DataLoader(
    test_dataset,
    batch_size=8,
    shuffle=False
)

print(
    "\nNumber of test batches:",
    len(test_loader)
)



# 6 Generate predictions on the held-out test set


def evaluate_model(
    model,
    loader,
    device
):
    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():

        for batch in loader:

            input_ids = (
                batch["input_ids"]
                .to(device)
            )

            attention_mask = (
                batch[
                    "attention_mask"
                ]
                .to(device)
            )

            labels = (
                batch["labels"]
                .to(device)
            )

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask
            )

            preds = (
                outputs.logits
                .squeeze(-1)
            )

            predictions.extend(
                preds
                .cpu()
                .numpy()
            )

            targets.extend(
                labels
                .cpu()
                .numpy()
            )

    return (
        np.asarray(
            targets
        ),
        np.asarray(
            predictions
        )
    )


y_test, y_pred = evaluate_model(
    model,
    test_loader,
    device
)

print(
    "\nNumber of test observations:",
    len(y_test)
)

print(
    "First 10 actual values:"
)

print(
    y_test[:10]
)

print(
    "\nFirst 10 predictions:"
)

print(
    y_pred[:10]
)



# 7 Calculate final held-out test performance


def calculate_metrics(
    y_true,
    y_pred
):
    y_true = np.asarray(
        y_true,
        dtype=float
    )

    y_pred = np.asarray(
        y_pred,
        dtype=float
    )

    mae = np.mean(
        np.abs(
            y_true
            - y_pred
        )
    )

    rmse = np.sqrt(
        np.mean(
            (
                y_true
                - y_pred
            ) ** 2
        )
    )

    ss_res = np.sum(
        (
            y_true
            - y_pred
        ) ** 2
    )

    ss_tot = np.sum(
        (
            y_true
            - np.mean(y_true)
        ) ** 2
    )

    r2 = (
        1
        - (
            ss_res
            / ss_tot
        )
    )

    pearson = pearsonr(
        y_true,
        y_pred
    ).statistic

    spearman = spearmanr(
        y_true,
        y_pred
    ).statistic

    return {
        "MAE": mae,
        "RMSE": rmse,
        "R2": r2,
        "Pearson": pearson,
        "Spearman": spearman
    }


test_metrics = calculate_metrics(
    y_test,
    y_pred
)

print(
    "\nFINAL HELD-OUT TEST RESULTS"
)

print(
    "---------------------------"
)

print(
    f"MAE:      "
    f"{test_metrics['MAE']:.4f}"
)

print(
    f"RMSE:     "
    f"{test_metrics['RMSE']:.4f}"
)

print(
    f"R²:       "
    f"{test_metrics['R2']:.4f}"
)

print(
    f"Pearson:  "
    f"{test_metrics['Pearson']:.4f}"
)

print(
    f"Spearman: "
    f"{test_metrics['Spearman']:.4f}"
)



# 8 Add predictions and prediction errors


test[
    "Predicted_BT_Easiness"
] = y_pred

test[
    "Prediction_Error"
] = (
    test[
        "Predicted_BT_Easiness"
    ]
    - test[
        "BT.Easiness"
    ]
)

test[
    "Absolute_Error"
] = (
    test[
        "Prediction_Error"
    ]
    .abs()
)

print(
    "\nPrediction examples:"
)

print(
    test[
        [
            "BT.Easiness",
            "Predicted_BT_Easiness",
            "Prediction_Error",
            "Absolute_Error"
        ]
    ].head()
)



# 9 Plot actual versus predicted BT.Easiness


plt.figure(
    figsize=(7, 6)
)

plt.scatter(
    y_test,
    y_pred,
    alpha=0.5
)

minimum = min(
    y_test.min(),
    y_pred.min()
)

maximum = max(
    y_test.max(),
    y_pred.max()
)

plt.plot(
    [minimum, maximum],
    [minimum, maximum]
)

plt.xlabel(
    "Actual BT.Easiness"
)

plt.ylabel(
    "Predicted BT.Easiness"
)

plt.title(
    "Fine-tuned ReadaBERT: "
    "Actual vs Predicted Readability"
)

plt.tight_layout()

plt.savefig(
    ACTUAL_PREDICTED_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 10 Inspect the largest prediction errors


worst = test.sort_values(
    "Absolute_Error",
    ascending=False
)

print(
    "\nLargest prediction errors:"
)

print(
    worst[
        [
            "Excerpt",
            "BT.Easiness",
            "Predicted_BT_Easiness",
            "Absolute_Error"
        ]
    ].head(10)
)



# 11 Obtain training-set reference values for normalisation


train = pd.read_csv(
    TRAIN_PATH
)

train = train.dropna(
    subset=[
        "BT.Easiness"
    ]
).copy()

train[
    "BT.Easiness"
] = (
    train[
        "BT.Easiness"
    ]
    .astype(float)
)

E_MIN = (
    train[
        "BT.Easiness"
    ]
    .min()
)

E_MAX = (
    train[
        "BT.Easiness"
    ]
    .max()
)

print(
    "\nTraining BT.Easiness minimum:",
    E_MIN
)

print(
    "Training BT.Easiness maximum:",
    E_MAX
)



# 12 Convert predicted easiness into difficulty D


test[
    "Predicted_Easiness_Normalized"
] = (
    (
        test[
            "Predicted_BT_Easiness"
        ]
        - E_MIN
    )
    /
    (
        E_MAX
        - E_MIN
    )
)

test[
    "Predicted_Easiness_Normalized"
] = (
    test[
        "Predicted_Easiness_Normalized"
    ]
    .clip(
        0,
        1
    )
)

test[
    "Difficulty_D"
] = (
    1
    - test[
        "Predicted_Easiness_Normalized"
    ]
)

print(
    "\nDifficulty D distribution:"
)

print(
    test[
        "Difficulty_D"
    ]
    .describe()
)

print(
    "\nExample D values:"
)

print(
    test[
        [
            "BT.Easiness",
            "Predicted_BT_Easiness",
            "Difficulty_D"
        ]
    ].head()
)



# 13 Check whether any predictions required clipping


predicted_normalized_raw = (
    (
        test[
            "Predicted_BT_Easiness"
        ]
        - E_MIN
    )
    /
    (
        E_MAX
        - E_MIN
    )
)

n_clipped = (
    (
        predicted_normalized_raw
        < 0
    )
    |
    (
        predicted_normalized_raw
        > 1
    )
).sum()

print(
    "\nPredictions outside training range:",
    int(n_clipped)
)

print(
    "Percentage clipped:",
    round(
        n_clipped
        / len(test)
        * 100,
        2
    )
)


# 14 Plot the final textual-difficulty signal D


plt.figure(
    figsize=(7, 5)
)

plt.hist(
    test[
        "Difficulty_D"
    ],
    bins=20
)

plt.xlabel(
    "Predicted Difficulty D"
)

plt.ylabel(
    "Frequency"
)

plt.title(
    "Distribution of ReadaBERT "
    "Difficulty Signal"
)

plt.tight_layout()

plt.savefig(
    DIFFICULTY_PLOT,
    dpi=300,
    bbox_inches="tight"
)

plt.close()



# 15 Export final ReadaBERT test predictions and D


test.to_csv(
    OUTPUT_CSV,
    index=False
)

print(
    "\nSaved:",
    OUTPUT_CSV
)


# 16 Save final test metrics


METRICS_PATH = (
    BASE_DIR
    / "phase6_readabert_test_metrics_rerun.csv"
)

metrics_df = pd.DataFrame(
    [
        {
            "MAE":
                test_metrics[
                    "MAE"
                ],

            "RMSE":
                test_metrics[
                    "RMSE"
                ],

            "R2":
                test_metrics[
                    "R2"
                ],

            "Pearson":
                test_metrics[
                    "Pearson"
                ],

            "Spearman":
                test_metrics[
                    "Spearman"
                ]
        }
    ]
)

metrics_df.to_csv(
    METRICS_PATH,
    index=False
)

print(
    "Saved:",
    METRICS_PATH
)



# 17 Confirm Phase 6 completion


print(
    "\nPHASE 6 COMPLETE"
)

print(
    "Model:",
    MODEL_PATH
)

print(
    "Test predictions:",
    OUTPUT_CSV
)

print(
    "Test metrics:",
    METRICS_PATH
)

print(
    "Actual vs predicted plot:",
    ACTUAL_PREDICTED_PLOT
)

print(
    "Difficulty plot:",
    DIFFICULTY_PLOT
)

Test file exists: True
Training file exists: True
Model exists: True

Test shape: (709, 14)
                                             Excerpt  BT.Easiness
0  And outside before the palace a great garden w...    -1.054013
1  One day he had gone beyond any point which he ...    -1.238432
2  So off went Lionel to be made a King. He had n...    -0.188186
3  Father had been away in the country for three ...    -1.009999
4  It was very dark at first, perhaps because the...    -1.258699

Model loaded from: /Users/nataliakolbina/PycharmProjects/JupyterProject/Reproduce ReadMe+++/readabert_clear_finetuned_best_rerun
Device: mps

Number of test batches: 89

Number of test observations: 709
First 10 actual values:
[-1.0540134  -1.2384322  -0.18818587 -1.009999   -1.258699    0.6103189
 -0.58635944 -1.8579944  -0.2189568  -0.4747609 ]

First 10 predictions:
[-1.7327951  -0.7059713   0.13899252 -0.65768105 -0.91411686  0.32609543
 -0.18736775 -1.5624256  -0.08881596 -0.61708134]

FINAL HELD-OUT 